# Seq2Seq Chatbot — Colab Training Notebook

**Project:** Ubuntu Dialogue Corpus · Bidirectional LSTM Seq2Seq with Bahdanau Attention  
**Models trained:** Baseline (no attention) + Attention (Bahdanau) — sequentially  
**Runtime:** Google Colab GPU (A100 recommended for bfloat16 AMP)

## Quickstart
1. Upload your `nlp-chatbot-projectv2/` folder to Google Drive (keep the folder structure)
2. Set `MODE` in **Cell 1** to `"mini"` (first test) or `"full"` (final training)
3. Run all cells top to bottom

## Drive folder structure expected
```
MyDrive/nlp-chatbot-projectv2/
  new/
    config.py  dataset.py  models.py  train.py  evaluate.py  ...
    artifacts/           ← phase1 pipeline outputs (stage6_*.jsonl, stage8_*.npy, ...)
    artifacts_mini/      ← mini pipeline outputs (mini mode only)
    checkpoints/         ← created automatically
    tb_logs/             ← created automatically
    logs/                ← created automatically
```

> **Tip:** Run **Cell 5** first to check which artifacts are present before training.

In [ ]:
# ─── Cell 1: Mount Drive + select run mode ───────────────────────────────────
#
# MODE = "mini"  →  artifacts_mini/  |  50k train pairs  |  up to 20 epochs
#                   fast sanity check (~20 min on A100)
#
# MODE = "full"  →  artifacts/       |  full dataset     |  20 epochs
#                   final training   (~2–4 h on A100)
#
MODE = "full"   # ← change to "mini" for a quick sanity check

# ─── Drive path ───────────────────────────────────────────────────────────────
# Edit PROJECT_DIR if your Drive folder is named differently.
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/nlp-chatbot-projectv2'

import os
from pathlib import Path

if MODE == "mini":
    ARTIFACT_DIR = f'{PROJECT_DIR}/new/artifacts_mini'
    CKPT_DIR     = f'{PROJECT_DIR}/new/checkpoints_mini'
    TB_DIR       = f'{PROJECT_DIR}/new/tb_logs_mini'
else:
    ARTIFACT_DIR = f'{PROJECT_DIR}/new/artifacts'
    CKPT_DIR     = f'{PROJECT_DIR}/new/checkpoints'
    TB_DIR       = f'{PROJECT_DIR}/new/tb_logs'

LOG_DIR = f'{PROJECT_DIR}/new/logs'

for d in (ARTIFACT_DIR, CKPT_DIR, TB_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

print(f"MODE         = {MODE}")
print(f"PROJECT_DIR  = {PROJECT_DIR}")
print(f"ARTIFACT_DIR = {ARTIFACT_DIR}")
print(f"CKPT_DIR     = {CKPT_DIR}")
print()

if not Path(f'{PROJECT_DIR}/new/train.py').exists():
    raise FileNotFoundError(
        f"train.py not found at {PROJECT_DIR}/new/train.py\n"
        "Upload your nlp-chatbot-projectv2 folder to Drive first."
    )
print("✓ Project files found")


In [ ]:
# ─── Cell 2: Install / verify dependencies ───────────────────────────────────
# Only installs packages that are missing (speeds up re-runs).
import importlib, subprocess, sys

# torch is pre-installed on Colab; upgrade if version is old.
import torch
if tuple(int(x) for x in torch.__version__.split('.')[:2]) < (2, 1):
    print(f"torch {torch.__version__} is old — upgrading …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "--upgrade", "torch"])
    print("torch upgraded — kernel restart may be needed if import errors appear")

REQUIRED = {
    "sentencepiece": "sentencepiece>=0.1.99",
    "gensim":        "gensim>=4.3.0",
    "sacrebleu":     "sacrebleu>=2.3.1",
    "rouge_score":   "rouge-score>=0.1.2",
    "bert_score":    "bert-score>=0.3.13",
    "seaborn":       "seaborn>=0.12.0",
    "tensorboard":   "tensorboard>=2.12.0",
    "tqdm":          "tqdm>=4.65.0",
}

missing = [spec for pkg, spec in REQUIRED.items()
           if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"Installing {len(missing)} package(s): {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + missing)
    print("Installed ✓")
else:
    print("All dependencies present ✓")

print(f"torch {torch.__version__}  |  CUDA {torch.version.cuda or 'N/A'}")


In [ ]:
# ─── Cell 3: sys.path + smoke-test imports ────────────────────────────────────
import sys

NEW_DIR = f'{PROJECT_DIR}/new'
for p in (PROJECT_DIR, NEW_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

# Smoke-check core imports
from config import CONFIG, set_seed, get_tf_ratio
from dataset import build_dataloaders
from models  import build_model
import train as _train

print("Imports OK ✓")
print(f"  vocab_size   = {CONFIG['vocab_size']}")
print(f"  enc_hidden   = {CONFIG['enc_hidden_dim']}")
print(f"  dec_hidden   = {CONFIG['dec_hidden_dim']}")
print(f"  num_epochs   = {CONFIG['num_epochs']}")


In [ ]:
# ─── Cell 4: GPU info + CONFIG overrides ─────────────────────────────────────
import copy, torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1e9
    print(f"  GPU  : {props.name}")
    print(f"  VRAM : {vram:.1f} GB")
    if vram >= 38:
        print("  → A100 / H100 detected — bfloat16 AMP + large batch")
    elif vram >= 14:
        print("  → RTX 3090/4090 class — bfloat16 AMP OK")
    else:
        print("  → Smaller GPU — consider reducing batch_size if OOM")
else:
    print("  ⚠ No GPU — training will be very slow on CPU")

# ── Build active config (copy to avoid mutating the imported singleton) ────────
cfg = copy.deepcopy(CONFIG)

cfg.update({
    # Absolute Drive paths (override config.py relative paths)
    "artifact_dir":           ARTIFACT_DIR,
    "checkpoint_dir":         CKPT_DIR,
    "tensorboard_dir":        TB_DIR,
    "log_dir":                LOG_DIR,
    "spm_model_path":         f'{ARTIFACT_DIR}/stage5_spm.model',
    "embedding_matrix_path":  f'{ARTIFACT_DIR}/stage8_embedding_matrix.npy',
})

# ── MODE-specific overrides ───────────────────────────────────────────────────
if MODE == "mini":
    cfg["max_train_samples"] = 50_000   # subset of mini artifacts for speed
    cfg["num_epochs"]        = 20       # early stopping will terminate early
    cfg["patience"]          = 5        # stop after 5 non-improving epochs
    cfg["grad_accum_steps"]  = 1        # no accumulation at mini scale
    cfg["batch_size"]        = 128 if device.type == "cuda" else 32
    cfg["num_workers"]       = 2 if device.type == "cuda" else 0
    print("\nMINI MODE — overrides applied:")
    print(f"  max_train_samples = {cfg['max_train_samples']:,}")
    print(f"  num_epochs        = {cfg['num_epochs']}")
    print(f"  patience          = {cfg['patience']}")
else:
    # Full training hardware tuning
    cfg["batch_size"]       = 256 if device.type == "cuda" else 32
    cfg["grad_accum_steps"] = 2        # effective batch = 512
    cfg["num_workers"]      = 4 if device.type == "cuda" else 0
    cfg["patience"]         = 0        # disable early stopping for full run
    print("\nFULL MODE — overrides applied:")
    print(f"  batch_size        = {cfg['batch_size']}")
    print(f"  grad_accum_steps  = {cfg['grad_accum_steps']} (eff. batch = {cfg['batch_size']*cfg['grad_accum_steps']:,})")
    print(f"  num_workers       = {cfg['num_workers']}")

cfg["amp_dtype"] = "bfloat16" if (
    device.type == "cuda" and
    torch.cuda.get_device_capability(0)[0] >= 8  # Ampere+ for bfloat16
) else "float32"
print(f"  amp_dtype         = {cfg['amp_dtype']}")

set_seed(cfg["seed"])
print("\nCONFIG ready ✓")


In [ ]:
# ─── Cell 5: Check Phase 1 artifacts ─────────────────────────────────────────
# Training requires stage 6 JSONL files + stage 8 embedding matrix.
# Run phase1.py (or phase1_mini.py) locally and upload the artifacts folder.
from pathlib import Path

REQUIRED_FOR_TRAINING = {
    "stage5_spm.model":             "SentencePiece tokenizer model",
    "stage6_train_ids.jsonl":       "Training pairs (BPE encoded)",
    "stage6_val_ids.jsonl":         "Validation pairs (BPE encoded)",
    "stage6_test_ids.jsonl":        "Test pairs (BPE encoded)",
    "stage6_vocab.json":            "BPE vocabulary",
    "stage8_embedding_matrix.npy":  "FastText embedding matrix (16000, 300)",
    "stage6_idx2word.json":         "Index→word mapping for decode",
}

print(f"Checking artifacts in: {ARTIFACT_DIR}\n")
art = Path(ARTIFACT_DIR)
all_present = True
for fname, desc in REQUIRED_FOR_TRAINING.items():
    path = art / fname
    if path.exists():
        size = path.stat().st_size / 1e6
        print(f"  ✅ {fname:<40} ({size:.1f} MB)  — {desc}")
    else:
        print(f"  ❌ {fname:<40} MISSING  — {desc}")
        all_present = False

print()
if all_present:
    print("✅ All required artifacts present — ready to train")
else:
    print("⚠️  Some artifacts are missing.")
    print("   Run phase1.py (full) or phase1_mini.py (mini) locally,")
    print("   then upload the artifacts/ folder to Drive before training.")


In [ ]:
# ─── Cell 6 (OPTIONAL): Run Phase 1 pipeline inside Colab ───────────────────
# Only needed if you do NOT have pre-built artifacts on Drive.
# Phase 1 on the full corpus takes ~60–90 min on Colab CPU.
# For a mini run it takes ~10 min.
#
# Skip this cell if Cell 5 showed ✅ for all artifacts.

RUN_PHASE1 = False   # ← set True to run Phase 1 here

if RUN_PHASE1:
    from pathlib import Path
    REQUIRED = [
        "stage6_train_ids.jsonl", "stage6_val_ids.jsonl",
        "stage6_test_ids.jsonl",  "stage8_embedding_matrix.npy",
    ]
    missing = [f for f in REQUIRED if not (Path(ARTIFACT_DIR) / f).exists()]

    if not missing:
        print("✓ Artifacts already present — skipping Phase 1")
    else:
        print(f"Missing: {missing}")
        print(f"Running Phase 1 (MODE={MODE}) …")

        if MODE == "mini":
            from phase1_mini import main as _p1_main
            # Override artifact dir to Drive path
            import phase1 as _p1
            _p1.PHASE1_CONFIG["artifact_dir"] = ARTIFACT_DIR
            _p1_main()
        else:
            from phase1 import main as _p1_main
            import phase1 as _p1
            _p1.PHASE1_CONFIG["artifact_dir"] = ARTIFACT_DIR
            _p1_main()

        print("\nPhase 1 complete ✓")
else:
    print("Skipping Phase 1 (RUN_PHASE1=False)")


In [ ]:
# ─── Cell 7: TensorBoard (optional) ──────────────────────────────────────────
# Uncomment and run this cell to monitor training loss in real time.
# Launch BEFORE starting training in Cell 8 so the dashboard is ready.

# %load_ext tensorboard
# %tensorboard --logdir {TB_DIR}

print(f"TensorBoard log dir: {TB_DIR}")
print("Uncomment the two lines above to start TensorBoard")


In [ ]:
# ─── Cell 8: Train both models ───────────────────────────────────────────────
# Trains Baseline (no attention) first, then Attention (Bahdanau).
# Resumes automatically from last checkpoint if the cell is interrupted.
#
# Expected runtimes on A100 (full mode, 1.1M pairs, 20 epochs):
#   Baseline  : ~60–90 min
#   Attention : ~90–120 min (slightly slower due to attention computation)
#
# Checkpoints saved to:  {CKPT_DIR}/baseline_best.pt
#                        {CKPT_DIR}/attention_best.pt
#                        {CKPT_DIR}/{model}_last.pt  ← resume on Colab disconnect

print("=" * 70)
print(f"  TRAINING  — MODE={MODE}  device={device}")
print("=" * 70)
print(f"  artifact_dir   : {cfg['artifact_dir']}")
print(f"  checkpoint_dir : {cfg['checkpoint_dir']}")
print(f"  batch_size     : {cfg['batch_size']}  ×  grad_accum={cfg['grad_accum_steps']} = eff. {cfg['batch_size']*cfg['grad_accum_steps']}")
print(f"  num_epochs     : {cfg['num_epochs']}")
if MODE == "mini":
    print(f"  max_train_samples: {cfg['max_train_samples']:,}")
    print(f"  early stop patience: {cfg['patience']}")
print()

_train.main(cfg=cfg, script_name=f"train_{MODE}")

print("\n✅ Training complete")


In [ ]:
# ─── Cell 9: Evaluate both models ────────────────────────────────────────────
# Loads best checkpoints for both models, runs BLEU / ROUGE-L / BERTScore / Distinct.
# Outputs written to checkpoint_dir:
#   bleu_results.json
#   baseline_manual_samples.json
#   attention_manual_samples.json
#   attention_heatmap.png        (attention model only)

from evaluate import run_evaluation

print("=" * 70)
print(f"  EVALUATION  — MODE={MODE}")
print("=" * 70)

run_evaluation(
    checkpoint_dir = cfg["checkpoint_dir"],
    artifact_dir   = cfg["artifact_dir"],
    config         = cfg,
    device         = device,
)

print("\n✅ Evaluation complete")
print(f"   Results → {cfg['checkpoint_dir']}/bleu_results.json")


In [ ]:
# ─── Cell 10: Print evaluation results ───────────────────────────────────────
import json
from pathlib import Path

results_path = Path(cfg["checkpoint_dir"]) / "bleu_results.json"
if results_path.exists():
    results = json.load(open(results_path))
    for model_name, metrics in results.items():
        print(f"\n{'='*50}")
        print(f"  {model_name.upper()} MODEL")
        print(f"{'='*50}")
        for k, v in metrics.items():
            if isinstance(v, float):
                print(f"  {k:<20} {v:.4f}")
            else:
                print(f"  {k:<20} {v}")
else:
    print("bleu_results.json not found — run Cell 9 first")
